# Mosaic-then-detect vs per-hour catalogs

Compare the two catalog orders on the same Full-band deep images:

1. **Per-hour then merge** (already on disk) — `detect_sources` on each LST
   hour, then `merge_lst_metacatalog` → `metacatalog_lst_Full.parquet`.
2. **Mosaic then detect** (this notebook) — reproject all hours onto one
   NCP-centered SIN grid with `lwa_healpix.coadd_fits`, then
   `run_pybdsf_on_hdu`.

A single SIN projection is valid only inside 90° of `CRVAL`. Native OVRO-LWA
images are already ~one SIN hemisphere (3122 px × 0.037° ≈ 1 radian). The
mosaic uses the same pixel scale, centered at the north celestial pole, so
all RA at Dec ≥ 0° fall in the valid disk. Sources south of the equator in
the per-hour catalogs will not have a counterpart on this grid.

Requires `lwa-catalog[detect]`, `lwa-healpix`, and the Parquet tree from
`ovro_lwa_metacatalog.ipynb`.

In [1]:
from __future__ import annotations

from pathlib import Path

import astropy.units as u
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS

from lwa_catalog import CatalogLayout
from lwa_catalog.constants import GAUL_COLUMNS
from lwa_catalog.create import discover_fits_files, discovered_slots
from lwa_catalog.create.detect import DEFAULT_BDSF_KW, run_pybdsf_on_hdu
from lwa_catalog.io import read_lst_merged, write_table
from lwa_healpix import coadd_fits

# --- user configuration ---------------------------------------------------
FITS_ROOT = Path("/lustre/pipeline/exopipe/phase3/coadd/Run_MASTERS1_20241218-20250827")
FITS_GLOB = "??h_Full/*_I_deep_Taper_Robust+0.0_dewarped*fits"
CATALOG_DIR = Path("/fast/claw/metacatalog_coadd")  # existing per-hour / LST-merged Parquet
OUTPUT_DIR = Path("/fast/claw/mosaic_detect")  # mosaic FITS + mosaic catalog

BAND = "Full"
MIN_ELEVATION_DEG = 10.0  # blank each hour below this elevation before reproject; None = off

# Mosaic SIN tangent point. NCP puts all RA in the valid hemisphere.
MOSAIC_CRVAL_RA_DEG = 0.0
MOSAIC_CRVAL_DEC_DEG = 90.0

BDSF_KW = dict(DEFAULT_BDSF_KW)
# Optional: blank SIN pixels that fail a pix→sky→pix round-trip (slow).
# BDSF_KW["check_outsideuniv"] = True

REUSE_MOSAIC_FITS = True
REUSE_MOSAIC_CATALOG = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
layout = CatalogLayout(CATALOG_DIR)
MOSAIC_FITS = OUTPUT_DIR / f"mosaic_sin_{BAND}.fits"
MOSAIC_CATALOG = OUTPUT_DIR / f"sources_mosaic_sin_{BAND}.parquet"


## Discover hourly Full-band FITS

Same filename parsing as the metacatalog notebook. One file per LST hour.

In [2]:
found = discover_fits_files(FITS_ROOT, patterns=(FITS_GLOB,))
slots = discovered_slots(found)
hour_metas = [m for m in slots.values() if m.band == BAND]
hour_metas = sorted(hour_metas, key=lambda m: m.lst_hour)
paths = [m.path for m in hour_metas]
print(f"{len(paths)} {BAND} images: {[m.lst_hour for m in hour_metas]}")
for m in hour_metas[:3]:
    print(f"  {m.lst_hour}  {m.path.name}")


23 Full images: ['00h', '01h', '02h', '03h', '04h', '05h', '06h', '07h', '08h', '09h', '10h', '11h', '12h', '13h', '14h', '15h', '16h', '17h', '18h', '19h', '20h', '21h', '22h']
  00h  Full_I_deep_Taper_Robust+0.0_dewarped_aligned_coadd_LST00h_20250809-20250816_N002.fits
  01h  Full_I_deep_Taper_Robust+0.0_dewarped_aligned_coadd_LST01h_20241218-20241229_N008.fits
  02h  Full_I_deep_Taper_Robust+0.0_dewarped_aligned_coadd_LST02h_20241218-20250103_N010.fits


## SIN target header and beam

Pixel scale and frequency come from the first hourly image. `BMAJ`/`BMIN`/`BPA`
are the median of the hourly headers (degrees). Grid size matches the native
SIN disk (`n = 2 * (90° / |CDELT|) + 1`, clipped to the reference `NAXIS`).

In [3]:
def _restfreq_hz(header: fits.Header) -> float:
    for key in ("RESTFREQ", "RESTFRQ", "CRVAL3", "FREQ"):
        if key in header:
            return float(header[key])
    raise KeyError("No RESTFREQ/RESTFRQ/CRVAL3/FREQ in reference header")


def median_beam(fits_paths: list[Path]) -> tuple[float, float, float]:
    bmaj, bmin, bpa = [], [], []
    for path in fits_paths:
        hdr = fits.getheader(path)
        bmaj.append(float(hdr["BMAJ"]))
        bmin.append(float(hdr["BMIN"]))
        bpa.append(float(hdr.get("BPA", 0.0)))
    return float(np.median(bmaj)), float(np.median(bmin)), float(np.median(bpa))


def ncp_sin_header(ref_header: fits.Header) -> fits.Header:
    cdelt = abs(float(ref_header["CDELT2"]))
    n_disk = int(round(2.0 * (90.0 / cdelt) + 1.0))
    naxis = int(ref_header.get("NAXIS1", n_disk))
    naxis = min(naxis, n_disk) if naxis > 0 else n_disk
    hdr = fits.Header()
    hdr["NAXIS"] = 2
    hdr["NAXIS1"] = naxis
    hdr["NAXIS2"] = naxis
    hdr["CTYPE1"] = "RA---SIN"
    hdr["CTYPE2"] = "DEC--SIN"
    hdr["CRVAL1"] = MOSAIC_CRVAL_RA_DEG
    hdr["CRVAL2"] = MOSAIC_CRVAL_DEC_DEG
    hdr["CRPIX1"] = (naxis + 1) / 2.0
    hdr["CRPIX2"] = (naxis + 1) / 2.0
    hdr["CDELT1"] = -cdelt
    hdr["CDELT2"] = cdelt
    hdr["CUNIT1"] = "deg"
    hdr["CUNIT2"] = "deg"
    hdr["LONPOLE"] = 180.0
    hdr["RADESYS"] = ref_header.get("RADESYS", "FK5")
    hdr["EQUINOX"] = ref_header.get("EQUINOX", 2000.0)
    return hdr


ref_header = fits.getheader(paths[0])
target_header = ncp_sin_header(ref_header)
bmaj, bmin, bpa = median_beam(paths)
restfreq = _restfreq_hz(ref_header)
print(
    f"SIN grid {target_header['NAXIS1']}×{target_header['NAXIS2']}  "
    f"CDELT={target_header['CDELT2']} deg  "
    f"CRVAL=({target_header['CRVAL1']}, {target_header['CRVAL2']})"
)
print(f"median beam BMAJ={bmaj:.5f} BMIN={bmin:.5f} BPA={bpa:.3f} deg")
print(f"RESTFREQ={restfreq:.3f} Hz")


SIN grid 3122×3122  CDELT=0.037 deg  CRVAL=(0.0, 90.0)
median beam BMAJ=0.22723 BMIN=0.16335 BPA=44.314 deg
RESTFREQ=57200637.276 Hz


## Reproject and coadd

`coadd_fits` footprint-weights each hour onto `target_header`. Pixels with
zero weight or outside the SIN disk (angular separation from CRVAL > 90°)
are set to NaN so PyBDSF blanks them.

In [4]:
def mask_outside_sin_disk(data: np.ndarray, header: fits.Header) -> np.ndarray:
    wcs = WCS(header).celestial
    ny, nx = data.shape
    yy, xx = np.mgrid[:ny, :nx]
    sky = wcs.pixel_to_world(xx, yy)
    center = SkyCoord(
        ra=float(header["CRVAL1"]) * u.deg,
        dec=float(header["CRVAL2"]) * u.deg,
        frame=sky.frame.name,
    )
    sep = sky.separation(center).deg
    out = np.array(data, dtype=np.float32, copy=True)
    out[sep > 90.0] = np.nan
    return out


def mosaic_hdu(
    data: np.ndarray,
    weight: np.ndarray,
    header: fits.Header,
) -> fits.PrimaryHDU:
    hdr = header.copy()
    image = np.array(data, dtype=np.float32, copy=True)
    image[weight <= 0] = np.nan
    image = mask_outside_sin_disk(image, hdr)
    hdr["BMAJ"] = bmaj
    hdr["BMIN"] = bmin
    hdr["BPA"] = bpa
    hdr["RESTFREQ"] = restfreq
    hdr["RESTFRQ"] = restfreq
    hdr["BUNIT"] = ref_header.get("BUNIT", "JY/BEAM")
    hdr["OBJECT"] = f"OVRO-LWA {BAND} LST mosaic"
    return fits.PrimaryHDU(data=image, header=hdr)


if REUSE_MOSAIC_FITS and MOSAIC_FITS.is_file():
    print(f"Reusing mosaic {MOSAIC_FITS}")
    with fits.open(MOSAIC_FITS, memmap=True) as hdul:
        hdu = fits.PrimaryHDU(
            data=np.array(hdul[0].data, dtype=np.float32, copy=True),
            header=hdul[0].header.copy(),
        )
else:
    combined, weight = coadd_fits(
        paths,
        target_header=target_header,
        min_elevation=MIN_ELEVATION_DEG,
    )
    hdu = mosaic_hdu(combined, weight, target_header)
    hdu.writeto(MOSAIC_FITS, overwrite=True)
    print(
        f"Wrote {MOSAIC_FITS}  finite={np.isfinite(hdu.data).sum()} / {hdu.data.size}  "
        f"weight>0={(weight > 0).sum()}"
    )


Wrote /fast/claw/mosaic_detect/mosaic_sin_Full.fits  finite=7533308 / 9746884  weight>0=7533308


## PyBDSF on the mosaic

Same `DEFAULT_BDSF_KW` as per-hour detection. The HDU already has `BMAJ`/`BMIN`
and `RESTFREQ`. Catalog is written as Parquet next to the mosaic FITS.

In [5]:
if REUSE_MOSAIC_CATALOG and MOSAIC_CATALOG.is_file():
    mosaic_cat = pd.read_parquet(MOSAIC_CATALOG)
    print(f"Reusing catalog {MOSAIC_CATALOG}: {len(mosaic_cat)} sources")
else:
    table = run_pybdsf_on_hdu(hdu, bdsf_kw=BDSF_KW)
    if table is None or len(table) == 0:
        mosaic_cat = pd.DataFrame(columns=list(GAUL_COLUMNS) + ["BMAJ", "BMIN", "BPA", "band"])
    else:
        mosaic_cat = table.to_pandas()
        keep = [c for c in GAUL_COLUMNS if c in mosaic_cat.columns]
        mosaic_cat = mosaic_cat[keep].copy()
    mosaic_cat["BMAJ"] = bmaj
    mosaic_cat["BMIN"] = bmin
    mosaic_cat["BPA"] = bpa
    mosaic_cat["band"] = BAND
    mosaic_cat["source_file"] = MOSAIC_FITS.name
    write_table(mosaic_cat, MOSAIC_CATALOG)
    print(f"Wrote {MOSAIC_CATALOG}: {len(mosaic_cat)} sources")


stty: 'standard input': Inappropriate ioctl for device
stty: 'standard input': Inappropriate ioctl for device


--> Wrote FITS file '/tmp/tmpuh_sd6z7.gaul.fits'
Wrote /fast/claw/mosaic_detect/sources_mosaic_sin_Full.parquet: 2411 sources


## Compare to LST-merged per-hour catalog

Load `metacatalog_lst_{BAND}.parquet` from the existing detect-then-merge run.
Match within `max(BMAJ)` (same radius as `merge._associate_catalogs`).
The northern-hemisphere cut (`DEC >= 0`) is applied to the per-hour catalog
so the two lists cover the same SIN disk.

In [6]:
lst_merged = read_lst_merged(layout, BAND)
assert isinstance(lst_merged, pd.DataFrame)

north = lst_merged[np.isfinite(lst_merged["DEC"]) & (lst_merged["DEC"] >= 0.0)].copy()
mosaic_ok = mosaic_cat[np.isfinite(mosaic_cat["RA"]) & np.isfinite(mosaic_cat["DEC"])].copy()


def match_within_beam(left: pd.DataFrame, right: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    if left.empty or right.empty:
        return np.array([], dtype=int), np.array([], dtype=int)
    left_sc = SkyCoord(ra=left["RA"].to_numpy() * u.deg, dec=left["DEC"].to_numpy() * u.deg)
    right_sc = SkyCoord(ra=right["RA"].to_numpy() * u.deg, dec=right["DEC"].to_numpy() * u.deg)
    radius = float(max(left["BMAJ"].max(), right["BMAJ"].max())) * u.deg
    idx_right, idx_left, sep2d, _ = left_sc.search_around_sky(right_sc, radius)
    limits = np.maximum(
        left["BMAJ"].to_numpy()[idx_left],
        right["BMAJ"].to_numpy()[idx_right],
    )
    keep = sep2d.to(u.deg).value <= limits
    return idx_left[keep], idx_right[keep]


idx_n, idx_m = match_within_beam(north, mosaic_ok)
matched_north = np.unique(idx_n)
matched_mosaic = np.unique(idx_m)

n_north = len(north)
n_mosaic = len(mosaic_ok)
n_both = len(matched_north)
n_north_only = n_north - n_both
n_mosaic_only = n_mosaic - len(matched_mosaic)

print(f"LST-merged {BAND} (all Dec):     {len(lst_merged):6d}")
print(f"LST-merged {BAND} (Dec >= 0):    {n_north:6d}")
print(f"Mosaic PyBDSF:                   {n_mosaic:6d}")
print(f"Matched within beam:             {n_both:6d}")
print(f"Per-hour only (Dec >= 0):        {n_north_only:6d}")
print(f"Mosaic only:                     {n_mosaic_only:6d}")
if n_north:
    print(f"Recovery of northern per-hour:   {n_both / n_north:.3f}")
if n_mosaic:
    print(f"Purity vs northern per-hour:     {len(matched_mosaic) / n_mosaic:.3f}")

if n_both:
    # One mosaic hit per northern source (nearest).
    north_sc = SkyCoord(ra=north["RA"].to_numpy() * u.deg, dec=north["DEC"].to_numpy() * u.deg)
    mosaic_sc = SkyCoord(
        ra=mosaic_ok["RA"].to_numpy() * u.deg, dec=mosaic_ok["DEC"].to_numpy() * u.deg
    )
    idx, sep, _ = north_sc.match_to_catalog_sky(mosaic_sc)
    within = sep.deg <= np.maximum(north["BMAJ"].to_numpy(), mosaic_ok["BMAJ"].to_numpy()[idx])
    flux_n = north["Peak_flux"].to_numpy(dtype=float)[within]
    flux_m = mosaic_ok["Peak_flux"].to_numpy(dtype=float)[idx[within]]
    ratio = flux_m / flux_n
    finite = np.isfinite(ratio) & (flux_n > 0)
    print(
        f"Peak_flux mosaic/per-hour: median={np.nanmedian(ratio[finite]):.3f}  "
        f"p16={np.nanpercentile(ratio[finite], 16):.3f}  "
        f"p84={np.nanpercentile(ratio[finite], 84):.3f}"
    )


LST-merged Full (all Dec):       9258
LST-merged Full (Dec >= 0):      7763
Mosaic PyBDSF:                     2411
Matched within beam:               2355
Per-hour only (Dec >= 0):          5408
Mosaic only:                         24
Recovery of northern per-hour:   0.303
Purity vs northern per-hour:     0.990
Peak_flux mosaic/per-hour: median=0.695  p16=0.611  p84=0.828
